# 02 - Extract Cal-Adapt WRF holdings (s3://cadcat/wrf/ucla)

UCLA WRF dynamical downscaling: **hourly precipitation** (culvert/stormwater intensity),
**snow water equivalent** (snow/avalanche exposure, rain-on-snow), and **near-surface
wind**, subset to the Tahoe bbox from the 9 km d02 domain.

Layout (verified 2026-07-29): one consolidated hourly Zarr store per
`<model>/<scenario>/1hr/all/<domain>` holding all variables on a curvilinear
Lambert-conformal grid. **Scenarios: historical + ssp370 only** - no WRF ssp245 runs
exist; document this asymmetry wherever WRF-derived metrics appear next to LOCA2.

- Wind `U10`/`V10` and SWE `SNOW` are aggregated to daily on extract.
- `RAINC`+`RAINNC` are run-accumulated totals: hourly rate = time-diff, clipped at 0
  (bucket resets), kept hourly. Hourly volume is why `precip_hourly.models` is a
  3-model subset in config.

The LOCA2-Hybrid CA product also lives in this bucket but is disabled by default -
it is the same 1/16 deg grid as the primary LOCA2 pull, CA-only (see METHODS.md).

In [1]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("02_extract_caladapt")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

Python: C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe


2026-07-31 16:09:56 | INFO | 02_extract_caladapt | Log file: C:\Users\mbindl\Documents\GitHub\PROTECT\climate\logs\02_extract_caladapt_2026-07-31_160956.log


2026-07-31 16:09:56 | INFO | 02_extract_caladapt | bbox: lon -120.5..-119.5, lat 38.5..39.5


In [2]:
import s3fs

CAD = cfg["sources"]["caladapt"]
WRF = CAD["wrf"]
FS = s3fs.S3FileSystem(anon=True)

cad_dir = RAW / "caladapt"
cad_dir.mkdir(exist_ok=True)

# discover which model/scenario stores exist
available = {}
for model in WRF["models"]:
    try:
        scens = [k.rsplit("/", 1)[1] for k in FS.ls(f"{WRF['s3_prefix']}/{model}")]
    except FileNotFoundError:
        log.warning(f"[wrf] no S3 path for model {model}")
        continue
    available[model] = [s for s in scens if s in ("historical", "ssp245", "ssp370")]
log.info(f"[wrf] stores: { {m: s for m, s in available.items()} }")


def zopen(store):
    """Open a Zarr store; fall back when consolidated .zmetadata is absent."""
    try:
        return xr.open_zarr(FS.get_mapper(store), consolidated=True)
    except KeyError:
        return xr.open_zarr(FS.get_mapper(store), consolidated=False)


# cadcat WRF has TWO publishing layouts:
#  A) raw all-variable hourly store:  <model>/<scen>/1hr/all/<domain>  (access-cm2, fgoals-g3)
#  B) tidy per-variable stores:       <model>/<scen>/<freq>/<var>/<domain>  (ec-earth3,
#     miroc6, mpi-esm1-2-hr) - includes a day frequency with wspd10mean/max, snow, prec
def layout_of(model, scen):
    return "all" if FS.exists(f"{WRF['s3_prefix']}/{model}/{scen}/1hr/all") else "pervar"


def wrf_bbox(ds):
    """Curvilinear subset: compute the 2-D lat/lon mask eagerly (xarray cannot index
    with a lazy dask boolean), then slice the enclosing x/y index window."""
    lat = ds.lat.compute()
    lon = ds.lon.compute()
    mask = ((lat >= BBOX["lat_min"]) & (lat <= BBOX["lat_max"])
            & (lon >= BBOX["lon_min"]) & (lon <= BBOX["lon_max"])).values
    if not mask.any():
        raise ValueError("bbox does not intersect the WRF domain")
    yy, xx = np.where(mask)
    ydim, xdim = ds.lat.dims
    return ds.isel({ydim: slice(yy.min(), yy.max() + 1),
                    xdim: slice(xx.min(), xx.max() + 1)})


def save_with_manifest(dset, out, store):
    dset.to_netcdf(out, encoding={v: {"zlib": True, "complevel": 4}
                                  for v in dset.data_vars})
    append_manifest({"file": str(out.relative_to(ROOT)), "source_url": f"s3://{store}",
                     "size_bytes": out.stat().st_size, "sha256": sha256_file(out),
                     "retrieved_date": str(pd.Timestamp.today().date()),
                     "notebook": "02_extract_caladapt"})

2026-07-31 16:10:28 | INFO | 02_extract_caladapt | [wrf] stores: {'access-cm2': ['historical', 'ssp370'], 'ec-earth3': ['historical', 'ssp370'], 'fgoals-g3': ['historical', 'ssp370'], 'miroc6': ['historical', 'ssp370'], 'mpi-esm1-2-hr': ['historical', 'ssp370']}


## Daily wind and SWE
Every configured model x available scenario: subset, aggregate hourly to daily
(mean + max for wind components, daily mean for SWE), save one NetCDF per group.

In [3]:
for model, scens in available.items():
    for scen in scens:
        out_wind = cad_dir / f"wrf_{model}_{scen}_wind_daily__tahoe.nc"
        out_swe = cad_dir / f"wrf_{model}_{scen}_swe_daily__tahoe.nc"
        if (out_wind.exists() and out_swe.exists()
                and not cfg["run"]["overwrite_downloads"]):
            log.info(f"[wrf] {model}/{scen} wind+swe present - skipped")
            continue
        try:
            if layout_of(model, scen) == "all":
                store = f"{WRF['s3_prefix']}/{model}/{scen}/1hr/all/{WRF['domain']}"
                sub = wrf_bbox(zopen(store)[WRF["wind"]["variables"]
                                            + WRF["swe"]["variables"]])
                wind = xr.Dataset({
                    "U10_mean": sub["U10"].resample(time="1D").mean(),
                    "V10_mean": sub["V10"].resample(time="1D").mean(),
                    "wspd_max": np.hypot(sub["U10"], sub["V10"]).resample(time="1D").max(),
                }).load()
                swe = sub["SNOW"].resample(time="1D").mean() \
                    .to_dataset(name="swe").load()
            else:
                base = f"{WRF['s3_prefix']}/{model}/{scen}/day"
                store = f"{base}/wspd10mean/{WRF['domain']}"
                wm = wrf_bbox(zopen(store))
                wx = wrf_bbox(zopen(f"{base}/wspd10max/{WRF['domain']}"))
                wind = xr.Dataset({
                    "wspd_mean": wm[list(wm.data_vars)[0]],
                    "wspd_max": wx[list(wx.data_vars)[0]],
                }).load()
                sn = wrf_bbox(zopen(f"{base}/snow/{WRF['domain']}"))
                swe = sn[list(sn.data_vars)[0]].to_dataset(name="swe").load()

            save_with_manifest(wind, out_wind, store)
            save_with_manifest(swe, out_swe, store)
            log.info(f"[wrf] {model}/{scen} ({layout_of(model, scen)}): "
                     f"wind {out_wind.stat().st_size/1e6:.1f} MB, "
                     f"swe {out_swe.stat().st_size/1e6:.1f} MB")
        except Exception as e:
            log.warning(f"[wrf] FAILED {model}/{scen} wind/swe: {e}")

2026-07-31 16:10:28 | INFO | 02_extract_caladapt | [wrf] access-cm2/historical wind+swe present - skipped


2026-07-31 16:10:28 | INFO | 02_extract_caladapt | [wrf] access-cm2/ssp370 wind+swe present - skipped


2026-07-31 16:11:58 | INFO | 02_extract_caladapt | [wrf] ec-earth3/historical (pervar): wind 16.0 MB, swe 3.5 MB


2026-07-31 16:13:27 | INFO | 02_extract_caladapt | [wrf] ec-earth3/ssp370 (pervar): wind 41.1 MB, swe 6.2 MB


2026-07-31 16:13:27 | INFO | 02_extract_caladapt | [wrf] fgoals-g3/historical wind+swe present - skipped


2026-07-31 16:13:27 | INFO | 02_extract_caladapt | [wrf] fgoals-g3/ssp370 wind+swe present - skipped


2026-07-31 16:15:10 | INFO | 02_extract_caladapt | [wrf] miroc6/historical (pervar): wind 16.0 MB, swe 3.4 MB


2026-07-31 16:16:41 | INFO | 02_extract_caladapt | [wrf] miroc6/ssp370 (pervar): wind 41.1 MB, swe 6.6 MB


2026-07-31 16:18:26 | INFO | 02_extract_caladapt | [wrf] mpi-esm1-2-hr/historical (pervar): wind 16.0 MB, swe 3.7 MB


2026-07-31 16:19:47 | INFO | 02_extract_caladapt | [wrf] mpi-esm1-2-hr/ssp370 (pervar): wind 41.2 MB, swe 7.7 MB


## Hourly precipitation
`precip_hourly.models` only (volume). Hourly rate = diff of (RAINC + RAINNC) along
time, negatives (accumulator resets) clipped to 0. First timestep is dropped by the
diff - one hour lost per run, irrelevant at climate scale.

In [4]:
for model in WRF["precip_hourly"]["models"]:
    for scen in available.get(model, []):
        out = cad_dir / f"wrf_{model}_{scen}_prec_hourly__tahoe.nc"
        if out.exists() and not cfg["run"]["overwrite_downloads"]:
            log.info(f"[wrf-prec] {model}/{scen} present - skipped")
            continue
        try:
            if layout_of(model, scen) == "all":
                store = f"{WRF['s3_prefix']}/{model}/{scen}/1hr/all/{WRF['domain']}"
                sub = wrf_bbox(zopen(store)[WRF["precip_hourly"]["variables"]])
                total = sub["RAINC"] + sub["RAINNC"]
                hourly = total.diff("time").clip(min=0) \
                    .to_dataset(name="prec_mm_hr").load()
            else:
                # per-variable layout ships hourly precip directly (already a rate)
                store = f"{WRF['s3_prefix']}/{model}/{scen}/1hr/prec/{WRF['domain']}"
                sub = wrf_bbox(zopen(store))
                da = sub[list(sub.data_vars)[0]]
                log.info(f"[wrf-prec] {model}/{scen} native units: "
                         f"{da.attrs.get('units', '?')}")
                hourly = da.to_dataset(name="prec_mm_hr").load()
            hourly["prec_mm_hr"].attrs["units"] = "mm/hr"
            save_with_manifest(hourly, out, store)
            log.info(f"[wrf-prec] {model}/{scen}: {out.stat().st_size/1e6:.1f} MB")
        except Exception as e:
            log.warning(f"[wrf-prec] FAILED {model}/{scen}: {e}")

log.info("Cal-Adapt WRF extract complete")

2026-07-31 16:19:47 | INFO | 02_extract_caladapt | [wrf-prec] access-cm2/historical present - skipped


2026-07-31 16:19:47 | INFO | 02_extract_caladapt | [wrf-prec] access-cm2/ssp370 present - skipped


2026-07-31 16:19:54 | INFO | 02_extract_caladapt | [wrf-prec] ec-earth3/historical native units: mm


2026-07-31 16:22:27 | INFO | 02_extract_caladapt | [wrf-prec] ec-earth3/historical: 37.4 MB


2026-07-31 16:22:36 | INFO | 02_extract_caladapt | [wrf-prec] ec-earth3/ssp370 native units: mm


...

2026-07-31 16:24:24 | INFO | 02_extract_caladapt | [wrf-prec] miroc6/historical native units: mm


2026-07-31 16:26:39 | INFO | 02_extract_caladapt | [wrf-prec] miroc6/historical: 36.8 MB


2026-07-31 16:26:50 | INFO | 02_extract_caladapt | [wrf-prec] miroc6/ssp370 native units: mm


2026-07-31 16:28:27 | INFO | 02_extract_caladapt | [wrf-prec] miroc6/ssp370: 84.5 MB


2026-07-31 16:28:27 | INFO | 02_extract_caladapt | Cal-Adapt WRF extract complete
